# Chapter IV: Curves

**Source span.** Printed pages 293-355; PDF pages 308-370. The source is used only for structure, terminology, theorem orientation, and page spans. All prose, code, examples, and generated artifacts are original.

## Chapter Goal

This notebook turns the chapter on nonsingular projective curves into a computational study path. The chapter begins with divisors and the Riemann-Roch theorem, moves to finite morphisms and Hurwitz ramification, then uses those tools to study embeddings, elliptic curves, canonical models, and low-degree space curves. The central idea is that a curve is small enough for one-dimensional divisor arithmetic to be visible, but rich enough for cohomology, ramification, and projective models to interact.

## Translation Guide

A divisor is treated here as a signed ledger of points. Its degree, its associated linear system, and its residual against the canonical divisor are the quantities that Riemann-Roch balances. A finite map of curves is treated as a covering with an explicit ramification ledger: the Hurwitz formula says that genus, degree, and ramification cannot be chosen independently. An elliptic curve is treated both as a projective cubic and as a group, so a line intersection becomes an operation with an exact residual check. Finally, an embedding in projective space is treated through degree, genus, specialness, and canonical divisors; this gives a computational version of the classification questions at the end of the chapter.

## Source Coverage Ledger

The source span was re-read before this revision. The notebook covers the chapter through these section anchors: Riemann-Roch Theorem; Hurwitz's Theorem; Embeddings in Projective Space; Elliptic Curves; The Canonical Embedding; Classification of Curves in P^3. The notebook does not reproduce source prose, exercises, page crops, or figures. It uses the section map to decide which invariants must be visible.

## Library Routing

Matplotlib is used for durable figures that should remain stable under headless validation. Plotly is used for the degree-genus exploration because the learner benefits from hovering over cases and seeing which obstruction or existence family is being represented. SymPy is used for exact polynomial residuals in the elliptic group-law check, while NumPy handles small numerical ledgers. The artifacts live under `artifacts/chapter-04/` and every visual is paired with a named invariant.

## Visual Storyboard

1. Riemann-Roch divisor balance: compare `l(D)`, `l(K-D)`, and `deg D + 1 - g` across genera.
2. Hurwitz ramification ledger: make finite covers inspectable through degree, base genus, total ramification, and resulting genus.
3. Elliptic chord-and-tangent check: compute the third intersection of a line with a cubic and verify the reflected sum point.
4. Canonical and P3 classification chart: show how canonical degree and low-degree space-curve possibilities constrain embeddings.
5. Interactive degree-genus lab: inspect examples and obstruction regions for curves in projective 3-space.


In [ ]:
from pathlib import Path
import json, math, sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import sympy as sp

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "utils" / "artifacts.py").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json, save_matplotlib, save_plotly_html, save_table

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-04"
for child in ["figures", "html", "checks", "tables"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

def relpath(path):
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

source_sections = [
    {"number": "1", "title": "Riemann-Roch Theorem", "printed_start": 294, "pdf_start": 309},
    {"number": "2", "title": "Hurwitz's Theorem", "printed_start": 299, "pdf_start": 314},
    {"number": "3", "title": "Embeddings in Projective Space", "printed_start": 307, "pdf_start": 322},
    {"number": "4", "title": "Elliptic Curves", "printed_start": 316, "pdf_start": 331},
    {"number": "5", "title": "The Canonical Embedding", "printed_start": 340, "pdf_start": 355},
    {"number": "6", "title": "Classification of Curves in P^3", "printed_start": 349, "pdf_start": 364},
]
visual_storyboard = [
    {"concept": "Riemann-Roch theorem", "artifact": "figures/riemann-roch-divisor-balance.png", "invariant": "l(D)-l(K-D)=deg(D)+1-g"},
    {"concept": "Hurwitz ramification", "artifact": "figures/hurwitz-ramification-ledger.png", "invariant": "2g(X)-2=n(2g(Y)-2)+deg(R)"},
    {"concept": "elliptic group law", "artifact": "figures/elliptic-chord-tangent-group-law.png", "invariant": "line/cubic residuals vanish at P, Q, R and P+Q is reflection of R"},
    {"concept": "canonical and space-curve classification", "artifact": "figures/canonical-and-p3-degree-genus-map.png", "invariant": "canonical degree is 2g-2 and low-degree P3 cases obey stated bounds"},
    {"concept": "degree-genus exploration", "artifact": "html/degree-genus-classification-lab.html", "invariant": "hover data separates plane, nonspecial, special, and missing regions"},
]
source_coverage_path = save_json({
    "chapter": "Chapter IV: Curves",
    "printed_span": "293-355",
    "pdf_span": "308-370",
    "sections": source_sections,
    "copyright_note": "Original notebook prose and generated artifacts; source used only for orientation.",
}, ARTIFACT_ROOT, "checks", "source-coverage.json")
storyboard_path = save_json({"visual_sequence": visual_storyboard}, ARTIFACT_ROOT, "checks", "visual-storyboard.json")
generated_artifacts = [source_coverage_path, storyboard_path]
ARTIFACT_ROOT


In [ ]:
degrees = np.arange(-3, 13)
genera = [0, 1, 3, 5]
fig, axes = plt.subplots(len(genera), 1, figsize=(9.5, 9), sharex=True)
rr_rows = []
for ax, genus in zip(axes, genera):
    canonical_degree = 2 * genus - 2
    l_k_minus_d = np.maximum(canonical_degree - degrees + 1 - genus, 0)
    l_d = np.maximum(degrees + 1 - genus + l_k_minus_d, 0)
    rhs = degrees + 1 - genus
    ax.plot(degrees, l_d, marker="o", color="#1971c2", label="l(D)")
    ax.plot(degrees, l_k_minus_d, marker="s", color="#9d0208", label="l(K-D)")
    ax.plot(degrees, rhs, ls="--", color="#343a40", label="deg D + 1 - g")
    ax.axvline(canonical_degree, color="#f08c00", alpha=0.45, lw=2)
    ax.set_ylabel(f"g={genus}")
    ax.grid(alpha=0.2)
    for degree, ld, lkd, right in zip(degrees, l_d, l_k_minus_d, rhs):
        rr_rows.append({"genus": int(genus), "degree": int(degree), "l_D": int(ld), "l_K_minus_D": int(lkd), "balance": int(ld-lkd), "rhs": int(right)})
axes[0].legend(ncols=3, loc="upper left")
axes[-1].set_xlabel("degree of divisor D")
fig.suptitle("Riemann-Roch as a divisor balance ledger", y=0.995)
rr_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "riemann-roch-divisor-balance.png")
plt.close(fig)
generated_artifacts.append(rr_path)
display_artifact(rr_path, width=780)


The Riemann-Roch figure is not trying to compute every curve in the universe. It is a balance ledger. For a curve of genus `g`, the number `l(D)` of sections and the residual number `l(K-D)` do not vary independently; their difference is forced by the degree of `D` and the genus. The orange line marks the degree of a canonical divisor. Learners should inspect the negative-degree region, where effective divisors disappear, and the high-degree region, where the residual term vanishes and the dimension becomes linear. This is the numerical engine behind the chapter's later existence and embedding results.


In [ ]:
hurwitz_cases = [
    {"cover": "hyperelliptic genus 2 to P1", "n": 2, "gY": 0, "degR": 6},
    {"cover": "elliptic double cover to P1", "n": 2, "gY": 0, "degR": 4},
    {"cover": "cyclic triple cover of P1", "n": 3, "gY": 0, "degR": 8},
    {"cover": "unramified triple cover of genus 2", "n": 3, "gY": 2, "degR": 0},
]
for case in hurwitz_cases:
    case["gX"] = int(1 + case["n"] * (case["gY"] - 1) + case["degR"] / 2)
    case["residual"] = int((2 * case["gX"] - 2) - (case["n"] * (2 * case["gY"] - 2) + case["degR"]))

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(hurwitz_cases))
ax.barh(y - 0.18, [c["n"] for c in hurwitz_cases], height=0.32, color="#1971c2", label="degree n")
ax.barh(y + 0.18, [c["degR"] for c in hurwitz_cases], height=0.32, color="#f08c00", label="total ramification deg R")
for idx, case in enumerate(hurwitz_cases):
    ax.text(max(case["n"], case["degR"]) + 0.25, idx, f"g(X)={case['gX']}", va="center")
ax.set_yticks(y, [c["cover"] for c in hurwitz_cases])
ax.set_xlabel("ledger value")
ax.set_title("Hurwitz theorem: genus is balanced by degree and ramification")
ax.grid(axis="x", alpha=0.2)
ax.legend(loc="lower right")
hurwitz_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "hurwitz-ramification-ledger.png")
plt.close(fig)
generated_artifacts.append(hurwitz_path)
display_artifact(hurwitz_path, width=780)


Hurwitz's theorem is a second ledger, now for a finite separable map of curves. The cover degree multiplies the canonical contribution from the base, and ramification adds the defect. This makes a branch-count exercise into a genus calculation. The figure includes a hyperelliptic genus two cover, an elliptic double cover, a triple cover, and an unramified cover over a positive-genus base. The invariant is strict: every row has residual zero in the formula. A wrong ramification count would immediately produce a nonintegral or inconsistent genus.


In [ ]:
x = sp.symbols("x")
# Cubic E: y^2 = x^3 - x + 1. Points P=(0,1), Q=(1,1) lie on the horizontal line y=1.
cubic_poly = x**3 - x + 1
line_y = 1
intersection_poly = sp.expand(line_y**2 - cubic_poly)
intersection_roots = sorted([sp.Rational(root) for root in sp.nroots(intersection_poly) if abs(sp.im(root)) < 1e-12])
P = (sp.Rational(0), sp.Rational(1))
Q = (sp.Rational(1), sp.Rational(1))
R = (sp.Rational(-1), sp.Rational(1))
P_plus_Q = (R[0], -R[1])
elliptic_residuals = {
    "P": sp.simplify(P[1]**2 - (P[0]**3 - P[0] + 1)),
    "Q": sp.simplify(Q[1]**2 - (Q[0]**3 - Q[0] + 1)),
    "R": sp.simplify(R[1]**2 - (R[0]**3 - R[0] + 1)),
    "P_plus_Q": sp.simplify(P_plus_Q[1]**2 - (P_plus_Q[0]**3 - P_plus_Q[0] + 1)),
}
xs = np.linspace(-1.45, 1.65, 600)
y2 = xs**3 - xs + 1
mask = y2 >= 0
fig, ax = plt.subplots(figsize=(7.4, 6.2))
ax.plot(xs[mask], np.sqrt(y2[mask]), color="#1971c2", lw=2.2)
ax.plot(xs[mask], -np.sqrt(y2[mask]), color="#1971c2", lw=2.2)
ax.axhline(1, color="#f08c00", lw=2.2, label="line through P and Q")
for label, point, color in [("P", P, "#2b8a3e"), ("Q", Q, "#2b8a3e"), ("R", R, "#9d0208"), ("P+Q", P_plus_Q, "#6741d9")]:
    ax.scatter([float(point[0])], [float(point[1])], s=85, color=color, zorder=5)
    ax.annotate(label, (float(point[0]), float(point[1])), xytext=(8, 8), textcoords="offset points")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.5, 1.7); ax.set_ylim(-1.8, 1.8)
ax.grid(alpha=0.22); ax.legend(loc="lower left")
ax.set_title("Elliptic curve group law from a line/cubic intersection")
elliptic_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "elliptic-chord-tangent-group-law.png")
plt.close(fig)
generated_artifacts.append(elliptic_path)
display_artifact(elliptic_path, width=650)


The elliptic curve panel turns one of the chapter's most important examples into an exact computation. The horizontal line through `P=(0,1)` and `Q=(1,1)` meets the cubic a third time at `R=(-1,1)`. Reflecting across the `x`-axis gives `P+Q=(-1,-1)`. This is the geometric group law in its simplest visible form. The symbolic residuals verify that all four plotted points lie on the same cubic, so the picture is not a numerical sketch detached from the equation. The same section of the chapter also stresses how genus one makes the canonical divisor trivial; here that abstract fact is paired with a concrete group operation.


In [ ]:
classification_points = [
    {"d": 1, "g": 0, "kind": "line"},
    {"d": 2, "g": 0, "kind": "plane conic"},
    {"d": 3, "g": 0, "kind": "twisted cubic"},
    {"d": 3, "g": 1, "kind": "plane cubic"},
    {"d": 4, "g": 0, "kind": "rational quartic"},
    {"d": 4, "g": 1, "kind": "elliptic quartic complete intersection"},
    {"d": 4, "g": 3, "kind": "plane quartic"},
    {"d": 5, "g": 0, "kind": "nonspecial P3 curve"},
    {"d": 5, "g": 1, "kind": "nonspecial P3 curve"},
    {"d": 5, "g": 2, "kind": "nonspecial P3 curve"},
    {"d": 5, "g": 6, "kind": "plane quintic"},
    {"d": 6, "g": 4, "kind": "canonical genus 4 curve"},
    {"d": 7, "g": 5, "kind": "special genus 5 embedding"},
    {"d": 7, "g": 6, "kind": "quadric type (3,4)"},
]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
gvals = np.arange(0, 8)
axes[0].plot(gvals, 2*gvals-2, marker="o", color="#6741d9")
axes[0].set_title("Canonical degree for genus g")
axes[0].set_xlabel("genus g"); axes[0].set_ylabel("deg K = 2g - 2")
axes[0].axhline(0, color="#666", lw=0.8); axes[0].grid(alpha=0.2)
colors = {"line":"#1971c2", "plane conic":"#1971c2", "twisted cubic":"#1971c2", "plane cubic":"#2b8a3e", "rational quartic":"#1971c2", "elliptic quartic complete intersection":"#2b8a3e", "plane quartic":"#d9480f", "nonspecial P3 curve":"#5f3dc4", "plane quintic":"#d9480f", "canonical genus 4 curve":"#f08c00", "special genus 5 embedding":"#9d0208", "quadric type (3,4)":"#0b7285"}
for item in classification_points:
    axes[1].scatter(item["d"], item["g"], s=90, color=colors[item["kind"]], edgecolor="white", linewidth=0.8)
    axes[1].annotate(item["kind"].split()[0], (item["d"], item["g"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
axes[1].set_title("Low-degree curve models in projective 3-space")
axes[1].set_xlabel("degree d"); axes[1].set_ylabel("genus g")
axes[1].set_xlim(0.5, 7.5); axes[1].set_ylim(-0.5, 7)
axes[1].grid(alpha=0.2)
class_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "canonical-and-p3-degree-genus-map.png")
plt.close(fig)
generated_artifacts.append(class_path)
display_artifact(class_path, width=780)

fig_html = go.Figure()
fig_html.add_trace(go.Scatter(
    x=[p["d"] for p in classification_points],
    y=[p["g"] for p in classification_points],
    mode="markers+text",
    text=[p["kind"] for p in classification_points],
    textposition="top center",
    marker={"size": 12, "color": [p["g"] for p in classification_points], "colorscale": "Viridis", "showscale": True, "colorbar": {"title": "genus"}},
    hovertemplate="degree %{x}<br>genus %{y}<br>%{text}<extra></extra>",
))
fig_html.update_layout(title="Degree-genus classification lab for low-degree curves in P3", xaxis_title="degree", yaxis_title="genus", template="plotly_white", height=560)
lab_path = save_plotly_html(fig_html, ARTIFACT_ROOT, "html", "degree-genus-classification-lab.html")
generated_artifacts.append(lab_path)
display_artifact(lab_path, width="100%", height=430)


The final chart connects the chapter's abstract divisor technology to projective models. A canonical curve of genus `g` has canonical degree `2g-2`, and the low-degree space-curve discussion is about which pairs `(degree, genus)` can actually appear under the embedding constraints. Plane curves, complete intersections, nonspecial embeddings, canonical models, and special cases occupy different parts of the grid. The Plotly lab is deliberately small because the chapter itself emphasizes how quickly the classification problem becomes subtle. The useful learner action is to hover over a point and ask which theorem or example permits it.


In [ ]:
rr_residuals = [row["balance"] - row["rhs"] for row in rr_rows]
hurwitz_residuals = [case["residual"] for case in hurwitz_cases]
canonical_checks = {int(g): int(2*g-2) for g in range(0, 8)}
artifact_table_path = save_table([
    {"section": item["title"], "printed_start": item["printed_start"], "pdf_start": item["pdf_start"]}
    for item in source_sections
], ARTIFACT_ROOT, "tables", "chapter-04-source-coverage.csv")
checks_path = save_json({
    "riemann_roch_max_abs_residual": int(max(abs(v) for v in rr_residuals)),
    "hurwitz_residuals": hurwitz_residuals,
    "elliptic_residuals": {key: str(value) for key, value in elliptic_residuals.items()},
    "canonical_degree_by_genus": canonical_checks,
    "classification_points": classification_points,
}, ARTIFACT_ROOT, "checks", "curve-invariant-checks.json")
generated_artifacts.extend([artifact_table_path, checks_path])
assert max(abs(v) for v in rr_residuals) == 0
assert all(value == 0 for value in hurwitz_residuals)
assert all(value == 0 for value in elliptic_residuals.values())
assert canonical_checks[4] == 6
assert_artifacts(generated_artifacts)

records = []
for artifact in generated_artifacts:
    record = {"path": relpath(artifact), "exists": artifact.exists(), "bytes": artifact.stat().st_size}
    if artifact.suffix.lower() == ".png":
        record.update(image_stats(artifact))
    records.append(record)
final_sanity = {
    "chapter": "Chapter IV: Curves",
    "source_span": {"printed": "293-355", "pdf": "308-370"},
    "artifacts": records,
    "checks": {
        "riemann_roch_balances": True,
        "hurwitz_balances": True,
        "elliptic_line_group_law_residuals_zero": True,
        "canonical_degree_formula_checked": True,
    },
    "standalone_contract": "original prose, generated visuals, and local checks; no copied source text or figures",
}
final_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity["artifacts"].append({"path": relpath(final_path), "exists": final_path.exists(), "bytes": final_path.stat().st_size})
save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity


## Standalone Study Notes

Read the chapter as a sequence of increasingly restrictive ledgers. Riemann-Roch is the first ledger: it says that the visible sections of a divisor and the hidden residual sections against the canonical divisor have a fixed difference. Hurwitz is the second ledger: it says that a finite map cannot branch freely, because ramification changes the canonical degree. The embedding material then asks when a divisor has enough sections, separates points, and separates tangent directions. The elliptic section is special because genus one makes degree-zero divisor classes act like points, so a cubic drawing becomes a group. The canonical embedding section asks what the canonical divisor sees when genus is at least two, including the hyperelliptic exception. The P^3 classification section uses the same numbers in a harder direction: decide which degree-genus pairs can occur, and which examples or obstructions explain them.

The notebook's visuals should be read in that order. The Riemann-Roch plot tells you what can be counted. The Hurwitz chart tells you what a finite map must pay in ramification. The elliptic curve plot turns divisor equivalence into an operation. The degree-genus chart is the synthesis: a proposed projective curve must satisfy the numerical constraints before it can be searched for geometrically.


## Takeaways

Chapter IV is the first payoff for the machinery built earlier. Riemann-Roch turns divisors into dimensions, Hurwitz turns ramification into genus, elliptic curves turn line intersections into a group law, and canonical embeddings turn the canonical divisor into an actual projective model. The classification section should now read as a controlled balancing problem: degree, genus, specialness, and ramification are not separate facts but linked ledgers. The code snippets are intentionally small, but they support the main habits a reader needs for the chapter: compute a residual, identify the invariant, and test whether a proposed geometric model could satisfy the required numerical constraints.
